# Mode 02: Z24 dataset.zip — tsai ResNet

This notebook trains a local tsai baseline from `raw_data/dataset.zip`. Run it with the working directory inside this project (for example `notebooks/modes02/local`). Its input remains in tsai's native `(samples, variables, sequence length)` layout: `(N, 27, 6000)`.

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import sys

start = Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents) if (p / 'src' / 'data' / 'z24_dataset_zip.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook with its working directory inside the shm project (for example notebooks/modes02/local)')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from fastai.metrics import accuracy
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from tsai.data.core import TSClassification
from tsai.data.validation import combine_split_data
from tsai.tslearner import TSClassifier
import tsai.inference

from src.data.z24_dataset_zip import (
    load_dataset_arrays, materialize_split, normalize_from_train,
    prepare_dataset_zip, split_by_setup,
)

print('Project root:', ROOT)

## Configuration

In [ ]:
MODEL_ARCH = 'ResNet'
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 1e-3
SEED = 42

## Validate, cache, and split

The same setup IDs are assigned to each split for every condition, so ten segments from one recording and a setup identity never cross split boundaries.

In [ ]:
CACHE_DIR = prepare_dataset_zip(ROOT)
inputs, labels = load_dataset_arrays(CACHE_DIR)
SPLITS, SETUP_SPLIT, METADATA = split_by_setup(labels, seed=SEED)

summary = []
for name, indexes in SPLITS.items():
    summary.append({
        'split': name,
        'segments': len(indexes),
        'recordings': len(set(METADATA['recording'][indexes].tolist())),
        'setups': SETUP_SPLIT[name].tolist(),
        'conditions': len(np.unique(labels[indexes])),
    })
display(pd.DataFrame(summary).set_index('split'))
print('Stored shape:', inputs.shape, inputs.dtype)

## Create tsai arrays and normalize from train only

In [ ]:
X_train, y_train = materialize_split(inputs, labels, SPLITS['train'], channels_last=False)
X_validation, y_validation = materialize_split(inputs, labels, SPLITS['validation'], channels_last=False)
X_test, y_test = materialize_split(inputs, labels, SPLITS['test'], channels_last=False)

X_train, X_validation, X_test, sensor_mean, sensor_std = normalize_from_train(
    X_train, X_validation, X_test, channels_last=False
)

for name, X_part, y_part in (
    ('train', X_train, y_train),
    ('validation', X_validation, y_validation),
    ('test', X_test, y_test),
):
    print(f'{name:10s}: X={X_part.shape}, y={y_part.shape}, memory={X_part.nbytes / 1024**2:.1f} MiB')
    assert X_part.shape[1:] == (27, 6000)
    assert X_part.dtype == np.float32
    assert set(np.unique(y_part)) == set(range(17))

In [ ]:
time_seconds = np.arange(6000) / 100.0
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for sensor in range(5):
    axes[0].plot(time_seconds, X_train[0, sensor], label=f'sensor_{sensor:02d}')
axes[0].set(title='One normalized training segment', xlabel='Time (seconds)', ylabel='Normalized value')
axes[0].grid(alpha=0.3)
axes[0].legend()
for name, y_part in (('train', y_train), ('validation', y_validation), ('test', y_test)):
    axes[1].plot(np.arange(17), np.bincount(y_part, minlength=17), marker='o', label=name)
axes[1].set(title='Segments per condition', xlabel='Label', ylabel='Segments')
axes[1].grid(alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()

## Build and train

Only train and validation are combined for the learner. Test remains outside training.

In [ ]:
X, y, tsai_splits = combine_split_data(
    [X_train, X_validation], [y_train, y_validation]
)
timestamp = datetime.now().strftime('%d-%m-%Y_%H-%M-%S')
ARTIFACT_DIR = ROOT / 'artifacts' / 'dataset_zip_tsai' / f'{MODEL_ARCH}_{timestamp}'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=False)

learner = TSClassifier(
    X, y, splits=tsai_splits, tfms=[None, TSClassification()],
    arch=MODEL_ARCH, metrics=accuracy, bs=BATCH_SIZE, wd=1e-4,
    path=ARTIFACT_DIR, model_dir='models', seed=SEED, verbose=True,
)
learner.summary()

In [ ]:
learner.fit_one_cycle(EPOCHS, LEARNING_RATE)

## Final test evaluation

In [ ]:
probabilities, _, _ = learner.get_X_preds(X_test, y_test, bs=16, with_decoded=True)
if hasattr(probabilities, 'detach'):
    probabilities = probabilities.detach().cpu().numpy()
else:
    probabilities = np.asarray(probabilities)
predictions = probabilities.argmax(axis=1).astype(np.int64)
y_test = np.asarray(y_test, dtype=np.int64).reshape(-1)

report = classification_report(
    y_test, predictions, labels=np.arange(17), output_dict=True, zero_division=0
)
test_accuracy = float(np.mean(predictions == y_test))
print(f'Test accuracy: {test_accuracy:.4f}')
display(pd.DataFrame(report).T.round(3))

matrix = confusion_matrix(y_test, predictions, labels=np.arange(17))
fig, ax = plt.subplots(figsize=(12, 12))
ConfusionMatrixDisplay(matrix, display_labels=np.arange(17)).plot(ax=ax, cmap='Blues', colorbar=False)
plt.show()

## Save reproducible outputs

In [ ]:
learner.export('z24_dataset_zip_tsai.pkl')
pd.DataFrame(report).T.to_csv(ARTIFACT_DIR / 'classification_report.csv')
np.savetxt(ARTIFACT_DIR / 'confusion_matrix.csv', matrix, fmt='%d', delimiter=',')
np.savez(
    ARTIFACT_DIR / 'preprocessing.npz',
    sensor_mean=sensor_mean, sensor_std=sensor_std,
    train_indexes=SPLITS['train'], validation_indexes=SPLITS['validation'], test_indexes=SPLITS['test'],
)
experiment = {
    'archive': 'raw_data/dataset.zip',
    'model_architecture': MODEL_ARCH,
    'data_contract': 'samples x sensors x time_samples',
    'train_shape': list(X_train.shape),
    'validation_shape': list(X_validation.shape),
    'test_shape': list(X_test.shape),
    'num_classes': 17,
    'seed': SEED,
    'setup_split': {name: values.tolist() for name, values in SETUP_SPLIT.items()},
    'sensor_mean': sensor_mean.tolist(),
    'sensor_std': sensor_std.tolist(),
    'test_accuracy': test_accuracy,
}
(ARTIFACT_DIR / 'experiment.json').write_text(json.dumps(experiment, indent=2), encoding='utf-8')
print('Saved to:', ARTIFACT_DIR)

In [ ]:
# Final train / validation / test metrics (macro average across classes)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def tsai_split_metrics(split_name, X, y):
    probabilities, _, _ = learner.get_X_preds(
        X, y, bs=16, with_decoded=True
    )
    if hasattr(probabilities, 'detach'):
        probabilities = probabilities.detach().cpu().numpy()
    else:
        probabilities = np.asarray(probabilities)
    predictions = probabilities.argmax(axis=1)
    targets = np.asarray(y).reshape(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        targets, predictions, average='macro', zero_division=0
    )
    return {
        'split': split_name,
        'accuracy': accuracy_score(targets, predictions),
        'precision_macro': precision,
        'recall_macro': recall,
        'f1_macro': f1,
    }

split_metrics = pd.DataFrame([
    tsai_split_metrics('train', X_train, y_train),
    tsai_split_metrics('validation', X_validation, y_validation),
    tsai_split_metrics('test', X_test, y_test),
]).set_index('split')
display(split_metrics.style.format('{:.2%}'))
split_metrics.to_csv(ARTIFACT_DIR / 'split_metrics.csv')